In [1]:
import radiate as rd
import polars as pl

rd.random.seed(67123)

### Create a fake dataset 

This dataset is a simple polynomial regression problem with a small it of noise

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
target_species = 4
species_threshold = 0.2

rolling_species_count = rd.Expr.select("species.count").rolling(25).mean()
err = rd.Expr.select("species.count").error(target_species) * 0.01

### Create the engine

This engine evolves `rd.Graph`s to fit the polynomial regression problem. The vertices of the graph are only allowed to use addition, multiplication, and constants. The engine also tracks the species count and species threshold as metrics, and uses the NEAT distance as a diversity metric.

In [4]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.linear()],
        edge=rd.Op.weight(),
        output=rd.Op.linear(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .diversity(
        rd.Dist.neat(),
        threshold=species_threshold,
        target=target_species,
    )
    .metrics(
        species_rolling=rolling_species_count,
        error=err,
    )
    .alter(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-10-12T16:55:22.720455Z  INFO State Change: PreStart -> Running
2026-10-12T16:55:22.724226Z  INFO Epoch 1    | Score:   2.0038 | Time: 3.01ms
2026-10-12T16:55:22.724532Z  INFO Epoch 2    | Score:   1.6821 | Time: 3.10ms
2026-10-12T16:55:22.724774Z  INFO Epoch 3    | Score:   1.6821 | Time: 3.30ms
2026-10-12T16:55:22.724925Z  INFO Epoch 4    | Score:   1.6821 | Time: 3.43ms
2026-10-12T16:55:22.725048Z  INFO Epoch 5    | Score:   1.6821 | Time: 3.54ms
2026-10-12T16:55:22.725160Z  INFO Epoch 6    | Score:   1.6821 | Time: 3.63ms
2026-10-12T16:55:22.725264Z  INFO Epoch 7    | Score:   1.6820 | Time: 3.72ms
2026-10-12T16:55:22.725368Z  INFO Epoch 8    | Score:   1.6820 | Time: 3.81ms
2026-10-12T16:55:22.725474Z  INFO Epoch 9    | Score:   1.6820 | Time: 3.91ms
2026-10-12T16:55:22.725588Z  INFO Epoch 10   | Score:   1.3108 | Time: 4.01ms
2026-10-12T16:55:22.725686Z  INFO Epoch 11   | Score:   1.3108 | Time: 4.09ms
2026-10-12T16:55:22.725799Z  INFO Epoch 12   | Score:   1.2533 | Time: 4.1

### Metrics

Using the `rd.MetricCollector` we can aggregate the metrics for the whole engine's run and visualize the metrics we want to track here once the engine has finished. In this case we only want to see our three expressions, but we could also see any `rd.Metric` that the engine tracks - refer to the docs for more details.

In [5]:
collector.plot("species.count", "species.threshold", "error")

In [6]:
df = collector.to_polars()
df.filter(pl.col("name") == "species.count").tail(20)
# df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""species.count""",6.0,3928.0,4.004077,2.313665,5.353045,3.114461,1.0,26.0,981,null,null,null,null,null,null,980,1,"[""species"", ""statistic""]"
"""species.count""",6.0,3934.0,4.00611,2.313362,5.351645,3.110063,1.0,26.0,982,null,null,null,null,null,null,981,1,"[""species"", ""statistic""]"
"""species.count""",5.0,3939.0,4.007121,2.312401,5.3472,3.109966,1.0,26.0,983,null,null,null,null,null,null,982,1,"[""species"", ""statistic""]"
"""species.count""",6.0,3945.0,4.009146,2.312098,5.345796,3.105587,1.0,26.0,984,null,null,null,null,null,null,983,1,"[""species"", ""statistic""]"
"""species.count""",5.0,3950.0,4.010152,2.311138,5.34136,3.105498,1.0,26.0,985,null,null,null,null,null,null,984,1,"[""species"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""species.count""",2.0,3991.0,4.007028,2.299869,5.289398,3.131575,1.0,26.0,996,null,null,null,null,null,null,995,1,"[""species"", ""statistic""]"
"""species.count""",3.0,3994.0,4.006018,2.298935,5.285104,3.134819,1.0,26.0,997,null,null,null,null,null,null,996,1,"[""species"", ""statistic""]"
"""species.count""",2.0,3996.0,4.004008,2.29866,5.283835,3.135745,1.0,26.0,998,null,null,null,null,null,null,997,1,"[""species"", ""statistic""]"
